# Scan metrology (C++)

The C++ counterpart of [`../python/67_scan_deviation.ipynb`](../python/67_scan_deviation.ipynb), [`68_scan_measurement.ipynb`](../python/68_scan_measurement.ipynb) and [`69_sections_to_sketches.ipynb`](../python/69_sections_to_sketches.ipynb) (roadmap §7): one feature fitted from a seed with sigma trimming (`operations/fit_primitives.hpp`), a spec of elements, dimensions and tolerances measured under the minimum-zone criterion (`operations/measure.hpp`), a scan's deviation from its nominal with its histogram (`operations/deviation.hpp`), wall thickness by ray, sphere and cone (`operations/thickness.hpp`), a chained section fitted to lines (`operations/slice.hpp`, `operations/fit_curves.hpp`), and decimation within a stated deviation (`operations/decimate.hpp`). The scans are synthetic, with a deterministic pseudo-noise so the stored outputs reproduce. See [measure](https://loumalouomega.github.io/meshioplusplus/measure) and [deviation](https://loumalouomega.github.io/meshioplusplus/deviation).

In [ ]:
#define MESHIOPLUSPLUS_IMPLEMENTATION
#include "../../src/single_include/meshioplusplus/meshioplusplus.hpp"
#include "mio_notebook.hpp"
#include <cmath>
#include <cstdio>
#include <iostream>
using namespace meshioplusplus;
using namespace mio_nb;

// A deterministic pseudo-noise in [-1, 1]: the same on every platform.
double noise(std::size_t i) {
    const double s = std::sin(12.9898 * double(i) + 78.233) * 43758.5453;
    return 2.0 * (s - std::floor(s)) - 1.0;
}

// A grid of quads on the parallelogram origin + [0,1] u + [0,1] v.
void add_face(std::vector<double>& rPts, std::vector<std::int64_t>& rQuads, const std::array<double, 3>& o,
              const std::array<double, 3>& u, const std::array<double, 3>& v, int nu, int nv) {
    const std::int64_t first = std::int64_t(rPts.size() / 3);
    for (int j = 0; j <= nv; ++j)
        for (int i = 0; i <= nu; ++i)
            for (int k = 0; k < 3; ++k) rPts.push_back(o[k] + u[k] * i / nu + v[k] * j / nv);
    for (int j = 0; j < nv; ++j)
        for (int i = 0; i < nu; ++i)
            for (int c : {j * (nu + 1) + i, j * (nu + 1) + i + 1, (j + 1) * (nu + 1) + i + 1, (j + 1) * (nu + 1) + i})
                rQuads.push_back(first + c);
}

Mesh make_quads(std::vector<double> pts, const std::vector<std::int64_t>& quads, double amplitude) {
    for (std::size_t i = 0; i < pts.size(); ++i) pts[i] += amplitude * noise(i);
    NDArray p = NDArray::Uninit(DType::Float64, {pts.size() / 3, std::size_t{3}});
    std::copy(pts.begin(), pts.end(), p.As<double>());
    NDArray c = NDArray::Uninit(DType::Int64, {quads.size() / 4, std::size_t{4}});
    std::copy(quads.begin(), quads.end(), c.As<std::int64_t>());
    Mesh m;
    m.AssignPoints(std::move(p));
    m.AddCellBlock("quad", std::move(c));
    return m;
}
// A 60 x 40 x 20 block of quads, its xmax face leaning by tiltDeg, one cell
// region per face, with pseudo-noise of the given amplitude.
Mesh scanned_block(double tiltDeg, double amplitude) {
    const double L = 60, W = 40, H = 20, d = H * std::tan(tiltDeg * M_PI / 180);
    struct F { const char* name; std::array<double, 3> o, u, v; };
    const F fs[] = {{"bottom", {0, 0, 0}, {0, W, 0}, {L, 0, 0}}, {"top", {0, 0, H}, {L + d, 0, 0}, {0, W, 0}},
                    {"xmin", {0, 0, 0}, {0, 0, H}, {0, W, 0}},   {"xmax", {L, 0, 0}, {0, W, 0}, {d, 0, H}},
                    {"ymin", {0, 0, 0}, {L, 0, 0}, {0, 0, H}},   {"ymax", {0, W, 0}, {0, 0, H}, {L, 0, 0}}};
    std::vector<double> pts;
    std::vector<std::int64_t> quads, counts;
    for (const F& f : fs) {
        const auto len = [](const std::array<double, 3>& a) { return std::sqrt(a[0] * a[0] + a[1] * a[1] + a[2] * a[2]); };
        const int nu = std::max(4, int(len(f.u) / 2)), nv = std::max(4, int(len(f.v) / 2));
        add_face(pts, quads, f.o, f.u, f.v, nu, nv);
        counts.push_back(std::int64_t(nu) * nv);
    }
    Mesh m = make_quads(pts, quads, amplitude);
    std::int64_t first = 0;
    for (std::size_t k = 0; k < 6; ++k) {
        NDArray ids = NDArray::Uninit(DType::Int64, {std::size_t(counts[k])});
        for (std::int64_t i = 0; i < counts[k]; ++i) ids.As<std::int64_t>()[i] = first + i;
        m.AddRegion(Region(fs[k].name, RegionKind::Cell, std::move(ids)));
        first += counts[k];
    }
    return m;
}
std::cout << "ready\n";

ready


## One feature from a seed

A half cylinder of radius 1.5, scanned with 0.002 of noise. `mSeeds` grows a region from the facet nearest the point while one primitive explains it, peels its rim and fits it as one patch; `FitUsedPoints::Sigma3` drops the points beyond three sigma before the final fit. The patch's row of `fit:patch_*` field data carries the fit, its form error and its sigma.

In [ ]:
std::vector<double> cpts;
std::vector<std::int64_t> cquads;
{
    const int n = 24, m = 8;
    for (int i = 0; i <= m; ++i)
        for (int k = 0; k <= n; ++k) {
            const double t = M_PI * k / n;
            cpts.insert(cpts.end(), {1.5 * std::cos(t), 1.5 * std::sin(t), 4.0 * i / m});
        }
    for (int i = 0; i < m; ++i)
        for (int k = 0; k < n; ++k)
            cquads.insert(cquads.end(), {i * (n + 1) + k, i * (n + 1) + k + 1, (i + 1) * (n + 1) + k + 1, (i + 1) * (n + 1) + k});
}
const Mesh half_cylinder = make_quads(cpts, cquads, 0.002);
FitPrimitivesOptions fo;
fo.mSeeds = {{{0.0, 1.5, 2.0}}};
fo.mTypes = 1u << static_cast<int>(FitPrimitiveType::Cylinder);
fo.mUsedPoints = FitUsedPoints::Sigma3;
const FitPrimitivesResult fit = fit_primitives(half_cylinder, fo);
const double* prm = fit.mMesh.FieldData("fit:patch_params").As<double>();
std::printf("cylinders %lld, feature cells %lld\n", (long long)fit.mNumCylinder, (long long)fit.mNumFeatureCells);
std::printf("axis (%.4f, %.4f, %.4f), radius %.5f\n", prm[3], prm[4], prm[5], prm[6]);
std::printf("form error %.5f, sigma %.5f\n", fit.mMesh.FieldData("fit:patch_form_error").As<double>()[0],
            fit.mMesh.FieldData("fit:patch_sigma").As<double>()[0]);
xcpp::display(render(fit.mMesh, "fit:type", "viridis"));

cylinders 1, feature cells 192
axis (0.0000, -0.0002, 1.0000), radius 1.49996
form error 0.00520, sigma 0.00116


## Elements, dimensions and tolerances

A 60 x 40 x 20 block, its `xmax` face leaning by 0.05 degrees, each face a cell region. The spec fits four planes, constructs their edge, and checks a height, a corner angle, a flatness, a perpendicularity and a parallelism under the minimum-zone criterion; each row reports PASS or FAIL.

In [ ]:
const Mesh block = scanned_block(0.05, 0.004);
const MeasureResult mr = measure(block, R"({
  "elements": [
    {"name": "A", "fit": "plane", "region": "bottom"},
    {"name": "B", "fit": "plane", "region": "xmax"},
    {"name": "T", "fit": "plane", "region": "top"},
    {"name": "edge", "construct": "intersection", "of": ["A", "B"]}],
  "dimensions": [
    {"name": "height", "type": "distance", "between": ["A", "T"], "nominal": 20, "tolerance": 0.01},
    {"name": "corner", "type": "angle", "between": ["A", "B"], "nominal": 90, "tolerance": 0.02}],
  "tolerances": [
    {"name": "flat_A", "type": "flatness", "feature": "A", "limit": 0.03},
    {"name": "perp_B", "type": "perpendicularity", "feature": "B", "datum": "A", "limit": 0.01},
    {"name": "par_T", "type": "parallelism", "feature": "T", "datum": "A", "limit": 0.05}],
  "criterion": "minimum_zone"})");
for (const MeasureCheck& c : mr.mDimensions) std::printf("%-8s %-17s %10.4f  %s\n", c.mName.c_str(), c.mType.c_str(), c.mValue, c.mPass ? "PASS" : "FAIL");
for (const MeasureCheck& c : mr.mTolerances) std::printf("%-8s %-17s %10.4f  %s\n", c.mName.c_str(), c.mType.c_str(), c.mValue, c.mPass ? "PASS" : "FAIL");
std::printf("failed rows: %lld\n", (long long)mr.mNumFail);

height   distance             20.0000  PASS
corner   angle                89.9498  FAIL
flat_A   flatness              0.0080  PASS
perp_B   perpendicularity      0.0249  FAIL
par_T    parallelism           0.0080  PASS
failed rows: 2


## Deviation from a nominal

The block's scan against its nominal (`scanned_block(0, 0)`: no lean, no noise): `deviation` writes `deviation:distance` and reports the statistics, the share within a tolerance and a histogram, drawn here with `histogram_chart`.

In [ ]:
const Mesh nominal = scanned_block(0.0, 0.0);
DeviationOptions dopt;
dopt.mTolerance = 0.01;
dopt.mBins = 16;
const DeviationResult dev = deviation(block, nominal, dopt);
std::printf("measured %lld of %lld points: min %.4f, max %.4f, rms %.4f, within +-0.01: %.3f\n",
            (long long)dev.mNumMeasured, (long long)dev.mNumPoints, dev.mMin, dev.mMax, dev.mRms, dev.mShareWithin);
xcpp::display(histogram_chart(dev.mHistogramEdges, dev.mHistogramCounts, "deviation from the nominal", "distance"));
xcpp::display(render(dev.mMesh, "deviation:distance", "coolwarm", true));

measured 2446 of 2446 points: min -0.0208, max 0.0217, rms 0.0042, within +-0.01: 0.951


## Wall thickness: ray, sphere and cone

A wedge plate, 1 thick at one end and 6 at the other, open along its sides. The ray reads a little more than the wall on the sloped face, the sphere (twice the largest empty sphere's radius) and the cone (the shortest of a fan of rays) read the wall.

In [ ]:
std::vector<double> wpts;
std::vector<std::int64_t> wquads;
add_face(wpts, wquads, {0, 0, 0}, {0, 20, 0}, {40, 0, 0}, 20, 40);
add_face(wpts, wquads, {0, 0, 1}, {40, 0, 5}, {0, 20, 0}, 40, 20);
const Mesh wedge = make_quads(wpts, wquads, 0.0);
for (ThicknessMethod method : {ThicknessMethod::Ray, ThicknessMethod::Sphere, ThicknessMethod::Cone}) {
    ThicknessOptions to;
    to.mMethod = method;
    const ThicknessResult t = thickness(wedge, to);
    std::printf("%-6s min %.4f  mean %.4f  max %.4f  missed %lld\n",
                method == ThicknessMethod::Ray ? "ray" : method == ThicknessMethod::Sphere ? "sphere" : "cone",
                t.mMin, t.mMean, t.mMax, (long long)t.mNumMissed);
}

ray    min 1.0625  mean 3.4824  max 5.9375  missed 20
sphere min 1.0543  mean 3.4555  max 5.8916  missed 20
cone   min 1.0548  mean 3.4876  max 5.9387  missed 0


## A section, fitted to lines

`SliceOptions::mChain` returns the section as ordered chains; `fit_curves` fits each with lines, arcs and circles within a tolerance. The block's section at half height is four lines (a closed polygon is never taken for a circle).

In [ ]:
SliceOptions so;
so.mOrigin = {{0.0, 0.0, 10.0}};
so.mChain = true;
const Mesh section = slice(block, so);
FitCurvesOptions fco;
fco.mTolerance = 0.02;
const FitCurvesResult curves = fit_curves(section, fco);
std::printf("%lld lines, %lld arcs, %lld circles, max residual %.4f\n", (long long)curves.mNumLines,
            (long long)curves.mNumArcs, (long long)curves.mNumCircles, curves.mMaxResidual);
for (const SketchPiece& p : curves.mPieces)
    std::printf("  %-6s (%.3f, %.3f) -> (%.3f, %.3f)\n", sketch_piece_type_name(p.mType), p.mStart[0], p.mStart[1], p.mEnd[0], p.mEnd[1]);

4 lines, 0 arcs, 0 circles, max residual 0.0044
  line   (-0.001, 0.000) -> (0.000, 40.001)
  line   (60.008, 39.997) -> (60.008, 0.001)
  line   (-0.003, 0.000) -> (59.997, -0.000)
  line   (-0.004, 40.001) -> (60.000, 39.998)


## Decimation within a deviation

`DecimateOptions::mMaxDeviation` bisects the face count for the coarsest mesh whose two-sided distance to the input stays within it for `mCoverage` of the samples.

In [ ]:
const Mesh ball = icosa_sphere(10.0, 4);
DecimateOptions dco;
dco.mMaxDeviation = 0.05;
const DecimateResult dr = decimate(ball, dco);
std::printf("%zu -> %zu triangles, deviation %.4f, met %s, %lld rounds\n", ball.Cells(0).NumCells(),
            dr.mMesh.Cells(0).NumCells(), dr.mDeviation, dr.mDeviationMet ? "yes" : "no", (long long)dr.mRounds);
xcpp::display(render(dr.mMesh));

5120 -> 1060 triangles, deviation 0.0476, met yes, 10 rounds
